# 13 Sorting

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Sort DataFrames with <code>orderBy</code> / <code>sort</code>, control direction and null placement, understand why a global sort is expensive, and know the cheaper alternatives: <code>sortWithinPartitions</code> and top-N.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Sorted output is needed for reports, top-N lists and "latest record" logic. But a global sort is one of the most expensive operations in Spark: it samples the data, shuffles everything into ordered ranges, then sorts each range. Knowing when <b>not</b> to sort is as important as knowing how.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How sorting works</b>
<ul><li><code>orderBy(...)</code> and <code>sort(...)</code> are the same: a <b>global</b> sort across all partitions</li><li>Default is ascending. Use <code>F.desc("col")</code>, <code>F.col("col").desc()</code> or <code>ascending=False</code></li><li><b>Nulls:</b> ascending puts nulls <b>first</b>, descending puts nulls <b>last</b>. Override with <code>asc_nulls_last()</code>, <code>desc_nulls_first()</code> and so on</li><li>A global sort uses <b>range partitioning</b>: Spark samples keys, picks boundaries, shuffles rows into ordered ranges, and sorts each range</li><li><code>sortWithinPartitions</code> sorts each partition independently, with <b>no shuffle</b></li><li><code>orderBy(...).limit(n)</code> is optimized into a <b>top-N</b> (<code>TakeOrderedAndProject</code>) that avoids sorting everything</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A marketplace wants the 10 best-selling products of the day out of 80 million order lines. Sorting all 80 million lines is wasteful. <code>groupBy(product).sum(qty)</code> reduces the data to about 200,000 products, then <code>orderBy(desc).limit(10)</code> is executed as a top-N where each partition keeps only its own top 10. Only a few hundred rows are compared at the end.
</div>

## Syntax

```python
from pyspark.sql import functions as F

df.orderBy("amount")                              # ascending
df.orderBy(F.col("amount").desc())                # descending
df.orderBy(F.desc("amount"), F.asc("name"))       # multiple keys
df.orderBy(["country", "amount"], ascending=[True, False])
df.orderBy(F.col("amount").desc_nulls_first())    # control nulls
df.sortWithinPartitions("amount")                 # no shuffle
df.orderBy(F.desc("amount")).limit(10)            # top-N
```

## Setup: sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a product sales DataFrame with ties and nulls.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
An 8-row table. <code>P6</code> has a null price and <code>P8</code> has a null category.
</div>

In [0]:
from pyspark.sql import functions as F

products = spark.createDataFrame(
    [
        ("P1", "Laptop", "Electronics", 1200.0, 30),
        ("P2", "Mouse", "Electronics", 25.0, 400),
        ("P3", "Desk", "Furniture", 350.0, 30),
        ("P4", "Chair", "Furniture", 150.0, 85),
        ("P5", "Novel", "Books", 15.0, 400),
        ("P6", "Mystery Box", "Toys", None, 12),
        ("P7", "Monitor", "Electronics", 300.0, 60),
        ("P8", "Gift Card", None, 50.0, 220),
    ],
    "product_id STRING, name STRING, category STRING, price DOUBLE, units_sold INT",
)
display(products)

## 1. Ascending and descending

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Sorts by price ascending, then descending, using three equivalent syntaxes.<br><br>
<b>Why it matters</b><br>You will see every style in real code. Note where the null price lands in each direction.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Ascending: <code>P6</code> (null) first, then 15.0 up to 1200.0. Descending: 1200.0 first and the null last. The three descending versions are identical.
</div>

In [0]:
products.orderBy("price").show()

products.orderBy(F.col("price").desc()).show()
products.orderBy(F.desc("price")).show()
products.orderBy("price", ascending=False).show()

## 2. Multiple keys and tie-breaking

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Sorts by <code>units_sold</code> descending, then by <code>name</code> ascending to break ties.<br><br>
<b>Why it matters</b><br>Without a tie-breaker, rows with equal keys can come back in a different order on each run. Deterministic output matters for tests, reports and "take the first row" logic.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>Mouse</code> and <code>Novel</code> (400 units) first, in alphabetical order, then <code>Gift Card</code> (220), <code>Chair</code> (85), and so on. <code>Desk</code> and <code>Laptop</code> (30) are also alphabetical.
</div>

In [0]:
products.orderBy(F.desc("units_sold"), F.asc("name")).show()

# The same with lists
products.orderBy(["units_sold", "name"], ascending=[False, True]).show()

## 3. Controlling null placement

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Overrides the default null position with <code>asc_nulls_last</code> and <code>desc_nulls_first</code>.<br><br>
<b>Why it matters</b><br>Reports usually want missing values at the end regardless of direction. SQL databases differ in their defaults, so being explicit avoids surprises when porting queries.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
First table: prices ascending with the null at the <b>end</b>. Second table: categories descending with the null category <b>first</b>.
</div>

In [0]:
products.orderBy(F.col("price").asc_nulls_last()).show()
products.orderBy(F.col("category").desc_nulls_first(), "name").show()

## 4. Custom sort order

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Sorts categories in a business-defined order (not alphabetical) using a <code>when</code> expression or <code>array_position</code>.<br><br>
<b>Why it matters</b><br>Business users often want an order like "Electronics, Furniture, Books, everything else". You sort by a computed rank.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Electronics rows first, then Furniture, then Books, then the rest. Both versions give the same order.
</div>

In [0]:
priority = F.when(F.col("category") == "Electronics", 1) \
            .when(F.col("category") == "Furniture", 2) \
            .when(F.col("category") == "Books", 3) \
            .otherwise(99)
products.orderBy(priority, "name").show()

# array_position returns the 1-based index of the value in the array (0 if not found).
order = F.array(*[F.lit(c) for c in ["Electronics", "Furniture", "Books"]])
rank = F.array_position(order, F.col("category"))
products.orderBy(F.when(rank == 0, 99).otherwise(rank), "name").show()

## 5. Top-N efficiently

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Gets the 3 best-selling products with <code>orderBy(...).limit(3)</code>, and looks at the physical plan.<br><br>
<b>Why it matters</b><br>Spark recognizes "sort + limit" and uses <code>TakeOrderedAndProject</code>: each partition keeps only its top N in a small heap, so the full data is never sorted. That is much cheaper than a full sort.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The 3 products with the highest <code>units_sold</code>, then a plan containing <code>TakeOrderedAndProject</code> (no full <code>Sort</code> over all rows).
</div>

In [0]:
products.orderBy(F.desc("units_sold"), "name").limit(3).show()

spark.range(1_000_000).withColumn("score", F.rand(seed=42)).orderBy(F.desc("score")).limit(5).explain()

## 6. sortWithinPartitions

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Sorts each partition separately and shows which partition each row is in.<br><br>
<b>Why it matters</b><br>When you write files, sorting <b>within</b> partitions (for example by a commonly filtered column) improves compression and data skipping without the cost of a global sort. The output as a whole is <b>not</b> globally sorted.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Rows grouped by <code>partition_id</code>, each group sorted by price, but the overall order is not sorted.
</div>

In [0]:
(
    products.repartition(3)                          # spread rows over 3 partitions
    .sortWithinPartitions("price")
    .withColumn("partition_id", F.spark_partition_id())
    .select("partition_id", "name", "price")
    .show()
)

## Under the hood

```
df.orderBy("amount")   -- global sort

  1. Sample the data to estimate the distribution of 'amount'   (an extra job)
  2. Choose range boundaries:  [min..100), [100..500), [500..max]
  3. Exchange rangepartitioning(amount)      <- SHUFFLE every row to its range
  4. Sort each partition locally
  => partition 0 < partition 1 < partition 2, each internally sorted = globally sorted

df.orderBy("amount").limit(10)   -- top-N
  TakeOrderedAndProject: each partition keeps its best 10, the driver merges them

df.sortWithinPartitions("amount")
  Sort per partition only, no Exchange
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Wide</span> `orderBy` shuffles **all** rows (range partitioning).

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Narrow</span> `sortWithinPartitions` doesn't shuffle.

Compare the two plans below: `Exchange rangepartitioning` appears only for the global sort.

In [0]:
data = spark.range(1_000_000).withColumn("amount", F.rand(seed=1) * 1000)

print("---- orderBy ----")
data.orderBy("amount").explain()

print("---- sortWithinPartitions ----")
data.sortWithinPartitions("amount").explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: sorting a huge DataFrame "just in case"</b><br>
A global sort shuffles all data. Sort only when the output order really matters, after aggregating, or use top-N with <code>limit</code>.<br><br>
<b>⛔ Problem: results come back in a different order each run</b><br>Ties in the sort key. Add a unique tie-breaker column.<br><br>
<b>⛔ Problem: nulls appear at the top of a descending report</b><br>They shouldn't by default, but they do appear first in ascending order. Use <code>asc_nulls_last()</code> / <code>desc_nulls_last()</code> explicitly.<br><br>
<b>⛔ Problem: order lost after a later transformation</b><br>A join, groupBy or repartition after <code>orderBy</code> can reorder rows. Sort as the <b>last</b> step.<br><br>
**⛔ Problem: <code>orderBy</code> then <code>dropDuplicates</code> doesn't keep the first row**<br>Order isn't preserved through the shuffle. Use a window with <code>row_number</code> (lessons 10 and 19).<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between <code>orderBy</code> and <code>sort</code>?</b><br>
None. They are aliases and both perform a global sort.<br><br>

<b>🎤 2. Why is a global sort expensive in Spark?</b><br>
Spark samples the data to choose range boundaries, then shuffles every row into a range partition, then sorts each partition. That is a full shuffle of all the data plus an extra sampling job.<br><br>

<b>🎤 3. What is <code>sortWithinPartitions</code> and when would you use it?</b><br>
It sorts each partition independently with no shuffle. I use it before writing files to cluster similar values together for better compression and data skipping, when global order isn't needed.<br><br>

<b>🎤 4. How does Spark optimize <code>orderBy().limit(n)</code>?</b><br>
It turns it into <code>TakeOrderedAndProject</code>. Each partition keeps only its top n rows, and the results are merged, so the full dataset is never sorted.<br><br>

<b>🎤 5. Where do nulls go when sorting?</b><br>
Ascending puts nulls first and descending puts them last by default. You can override that with <code>asc_nulls_last</code>, <code>desc_nulls_first</code> and the other variants.<br><br>

<b>🎤 6. Is the order of a DataFrame preserved after a join or groupBy?</b><br>
No. Any shuffle can reorder rows, so sorting should be the last step before output.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which query returns employees by salary from highest to lowest, with NULL salaries at the end?</b><br>
A. <code>ORDER BY salary DESC NULLS FIRST</code><br>
B. <code>ORDER BY salary DESC NULLS LAST</code><br>
C. <code>ORDER BY salary ASC NULLS LAST</code><br>
D. <code>SORT BY salary DESC</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> (In Spark SQL, <code>DESC</code> already defaults to NULLS LAST, but being explicit is clearest.) <code>SORT BY</code> sorts within partitions only.</details><br><br>

<b>Q2. What does <code>SORT BY</code> do in Spark SQL, compared to <code>ORDER BY</code>?</b><br>
A. They are identical<br>
B. <code>SORT BY</code> sorts within each partition. <code>ORDER BY</code> produces a total order<br>
C. <code>SORT BY</code> sorts in descending order by default<br>
D. <code>SORT BY</code> is not supported in Spark SQL<br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>SORT BY</code> is the SQL form of <code>sortWithinPartitions</code>.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b><br>
Using <code>products</code>:
<ol><li>Sort by category ascending (nulls last), then by price descending</li><li>Return the 2 cheapest products that have a known price</li><li>Sort with a custom category order: Books, Toys, Electronics, Furniture, then anything else</li><li>Return the top 3 products by revenue (<code>price * units_sold</code>), ignoring null prices</li><li>In one sentence: why should <code>orderBy</code> usually be the last step before writing or displaying?</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Two keys with null handling
products.orderBy(F.col("category").asc_nulls_last(), F.col("price").desc()).show()

# 2. Two cheapest with a known price
products.filter(F.col("price").isNotNull()).orderBy("price").limit(2).show()

# 3. Custom order via array_position
order = F.array(*[F.lit(c) for c in ["Books", "Toys", "Electronics", "Furniture"]])
pos = F.array_position(order, F.col("category"))
products.orderBy(F.when(pos == 0, 99).otherwise(pos), "name").show()

# 4. Top 3 by revenue
(
    products.filter(F.col("price").isNotNull())
    .withColumn("revenue", F.col("price") * F.col("units_sold"))
    .orderBy(F.desc("revenue"))
    .limit(3)
    .show()
)

# 5. Any later shuffle (join, groupBy, repartition) can reorder the rows, so sorting earlier is wasted work.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>orderBy</code> = <code>sort</code> = global sort, which shuffles everything with range partitioning</li><li>Ascending puts nulls first, descending puts nulls last. Override with <code>*_nulls_first/last</code></li><li>Add a tie-breaker for deterministic results</li><li><code>orderBy(...).limit(n)</code> becomes a cheap top-N (<code>TakeOrderedAndProject</code>)</li><li><code>sortWithinPartitions</code> sorts locally with no shuffle, which is good before writing files</li><li>Sort as the last step. Shuffles after it destroy the order</li><li>Custom orders: sort by a <code>when</code> rank or <code>array_position</code></li></ul>
</div>

| Task | Code |
|---|---|
| Ascending | `df.orderBy("x")` |
| Descending | `df.orderBy(F.desc("x"))` |
| Several keys | `df.orderBy(F.desc("a"), "b")` |
| Nulls last | `F.col("x").asc_nulls_last()` |
| Top N | `df.orderBy(F.desc("x")).limit(n)` |
| Per partition | `df.sortWithinPartitions("x")` |
| SQL | `ORDER BY x DESC NULLS LAST`, `SORT BY x` |

## Git commit

```
git add 01_spark_basics/13_sorting.ipynb
git commit -m "add 13 sorting notebook"
```